## Load dataset

In [7]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [8]:
fe_path = r"D:\NUS Note\IT5006\it5006_project\data\fe_data\features_delivery_time.csv"   # 修改成你的实际路径

df = pd.read_csv(fe_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())



Dataset shape: (96454, 72)

Columns:
['order_id', 'order_purchase_timestamp', 'purchase_month', 'customer_state', 'customer_region', 'customer_zip_code_prefix', 'customer_zip_lat', 'customer_zip_lng', 'customer_zip_density', 'customer_is_capital', 'remote_flag', 'order_value', 'mean_item_price', 'max_item_price', 'freight_value', 'item_count', 'unique_product_count', 'seller_count', 'seller_state_count', 'product_category_count', 'multi_seller_flag', 'product_category', 'primary_seller_state', 'primary_seller_region', 'region_pair', 'same_state', 'any_seller_same_state', 'same_city', 'any_seller_same_city', 'mean_seller_zip_lat', 'mean_seller_zip_lng', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'estimated_max_distance_km', 'geolocation_missing', 'total_weight_g', 'total_volume_cm3', 'max_item_weight_g', 'longest_side_cm', 'max_sum_of_dims_cm', 'volumetric_weight_kg', 'billable_weight_kg', 'density_g_cm3', 'payment_value', 'payment_installments', 'payment_type_count', 'p

## 1. Remove ID / Target / Split / Obvious duplicate variables

In [10]:
target = "delivery_days"

# exclude target and non-feature columns
exclude_cols = [
    "order_id",                    # ID
    "order_purchase_timestamp",    # 已经生成时间衍生变量
    "split",                       # Train/Val/Test标签
    "delivery_days"                # Target
]

# exclude duplicate month features
# keep
# purchase_month_sin
# purchase_month_cos
duplicate_cols = [
    "purchase_month",
    "purchase_month_of_year"
]

# exclude customer_zip_code_prefix
manual_drop_cols = exclude_cols + duplicate_cols + [
    "customer_zip_code_prefix"
]

# features after manual drop
feature_cols = [
    col for col in df.columns
    if col not in manual_drop_cols
]

print("Number of features after Step 1:", len(feature_cols))
print(feature_cols)

Number of features after Step 1: 65
['customer_state', 'customer_region', 'customer_zip_lat', 'customer_zip_lng', 'customer_zip_density', 'customer_is_capital', 'remote_flag', 'order_value', 'mean_item_price', 'max_item_price', 'freight_value', 'item_count', 'unique_product_count', 'seller_count', 'seller_state_count', 'product_category_count', 'multi_seller_flag', 'product_category', 'primary_seller_state', 'primary_seller_region', 'region_pair', 'same_state', 'any_seller_same_state', 'same_city', 'any_seller_same_city', 'mean_seller_zip_lat', 'mean_seller_zip_lng', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'estimated_max_distance_km', 'geolocation_missing', 'total_weight_g', 'total_volume_cm3', 'max_item_weight_g', 'longest_side_cm', 'max_sum_of_dims_cm', 'volumetric_weight_kg', 'billable_weight_kg', 'density_g_cm3', 'payment_value', 'payment_installments', 'payment_type_count', 'payment_type', 'used_voucher', 'payment_missing', 'promised_delivery_days', 'promised_da

## 2. Remove redundant variables with a correlation greater than 0.90.

In [ ]:
# Split the dataset into train, validation, and test sets

train_df = df[df["split"] == "train"].copy()

val_df = df[
    df["split"].isin(["validation", "val"])
].copy()

test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

X_train = train_df[feature_cols].copy()
y_train = train_df[target].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target].copy()

Train: (83947, 72)
Validation: (6156, 72)
Test: (6351, 72)


In [12]:
# check correlation > 0.90
# 只看 Train 中的数值变量

numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Number of numeric features:", len(numeric_cols))

corr_matrix = (
    X_train[numeric_cols]
    .corr()
    .abs()
)

upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs > 0.90
]

print("Highly correlated pairs (> 0.90):")
print(high_corr_pairs)

Number of numeric features: 58
Highly correlated pairs (> 0.90):
total_volume_cm3            volumetric_weight_kg          1.000000
estimated_min_distance_km   estimated_mean_distance_km    0.999548
estimated_mean_distance_km  estimated_max_distance_km     0.999546
estimated_min_distance_km   estimated_max_distance_km     0.998192
mean_item_price             max_item_price                0.998121
seller_past_orders          seller_past_deliveries        0.997864
order_value                 payment_value                 0.996257
same_state                  any_seller_same_state         0.994763
same_city                   any_seller_same_city          0.993858
seller_count                multi_seller_flag             0.973525
promised_delivery_days      weekend_days_to_estimate      0.970760
total_volume_cm3            billable_weight_kg            0.948528
volumetric_weight_kg        billable_weight_kg            0.948528
order_value                 max_item_price                0.9354

In [13]:
# mannually drop one of each pair of highly correlated features
corr_manual_drop = [
    "volumetric_weight_kg",

    "estimated_min_distance_km",
    "estimated_mean_distance_km",

    "max_item_price",

    "seller_past_deliveries",

    "payment_value",

    "same_state",
    "same_city",

    "multi_seller_flag",

    "weekend_days_to_estimate",

    "max_sum_of_dims_cm"
]

feature_cols_corr = [
    col for col in feature_cols
    if col not in corr_manual_drop
]

print("Features before correlation filtering:", len(feature_cols))
print("Features after correlation filtering:", len(feature_cols_corr))

print("\nDropped features:")
print(corr_manual_drop)

Features before correlation filtering: 65
Features after correlation filtering: 54

Dropped features:
['volumetric_weight_kg', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'max_item_price', 'seller_past_deliveries', 'payment_value', 'same_state', 'same_city', 'multi_seller_flag', 'weekend_days_to_estimate', 'max_sum_of_dims_cm']


## 3. LassoCV

### 3.1 Lasso preprocessing

In [17]:
# divide features into numeric and categorical
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("\nNumeric count:", len(numeric_features))
print("Categorical count:", len(categorical_features))

# numeric features
numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

# categorical features
categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

# define preprocessor
preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])


Numeric count: 58
Categorical count: 7


### 3.2 LassoCV

In [18]:
lasso_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "lasso",
        LassoCV(
            cv=5,
            random_state=42,
            max_iter=20000,
            n_alphas=100
        )
    )
])

lasso_pipeline.fit(
    X_train,
    y_train
)

best_alpha = (
    lasso_pipeline
    .named_steps["lasso"]
    .alpha_
)

print("Best alpha:", best_alpha)

Best alpha: 0.0038941461903852686


In [20]:
# check Lasso coefficients
feature_names = (
    lasso_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

lasso_coef = (
    lasso_pipeline
    .named_steps["lasso"]
    .coef_
)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": lasso_coef
})

coef_df["abs_coefficient"] = (
    coef_df["coefficient"].abs()
)

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

display(coef_df.head(10))

,feature,coefficient,abs_coefficient
55,num__seller_past_deliveries,-3.250768,3.250768
59,cat__customer_state_AL,3.095124,3.095124
54,num__seller_past_orders,3.077495,3.077495
78,cat__customer_state_RO,-2.882619,2.882619
83,cat__customer_state_SP,-2.453744,2.453744
148,cat__product_category_office_furniture,2.451211,2.451211
174,cat__primary_seller_state_PE,-2.366012,2.366012
68,cat__customer_state_MG,-2.350312,2.350312
73,cat__customer_state_PE,-1.890988,1.890988
87,cat__customer_region_NE,1.829908,1.829908


In [21]:
# filter features with non-zero coefficients
selected_encoded_features = coef_df[
    coef_df["coefficient"] != 0
].copy()

print(
    "Number of encoded features before Lasso:",
    len(coef_df)
)

print(
    "Number of non-zero features:",
    len(selected_encoded_features)
)

display(
    selected_encoded_features[
        [
            "feature",
            "coefficient",
            "abs_coefficient"
        ]
    ]
)

Number of encoded features before Lasso: 217
Number of non-zero features: 103


,feature,coefficient,abs_coefficient
55,num__seller_past_deliveries,-3.250768,3.250768
59,cat__customer_state_AL,3.095124,3.095124
54,num__seller_past_orders,3.077495,3.077495
78,cat__customer_state_RO,-2.882619,2.882619
83,cat__customer_state_SP,-2.453744,2.453744
...,...,...,...
57,num__seller_past_avg_freight,-0.013692,0.013692
35,num__payment_type_count,-0.013380,0.013380
8,num__freight_value,0.012757,0.012757
52,num__route_past_late_rate,-0.010091,0.010091


In [22]:
# validation MAE
val_pred_lasso = (
    lasso_pipeline.predict(X_val)
)

lasso_mae = mean_absolute_error(
    y_val,
    val_pred_lasso
)

lasso_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        val_pred_lasso
    )
)

lasso_r2 = r2_score(
    y_val,
    val_pred_lasso
)

print("Lasso Validation Performance")
print(f"MAE:  {lasso_mae:.3f}")
print(f"RMSE: {lasso_rmse:.3f}")
print(f"R²:   {lasso_r2:.3f}")

Lasso Validation Performance
MAE:  3.777
RMSE: 5.538
R²:   0.202


### Compare LASSO with full model performance

In [23]:
# linear
full_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LinearRegression()
    )
])

full_model.fit(
    X_train,
    y_train
)

val_pred_full = (
    full_model.predict(X_val)
)

full_mae = mean_absolute_error(
    y_val,
    val_pred_full
)

full_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        val_pred_full
    )
)

full_r2 = r2_score(
    y_val,
    val_pred_full
)

In [24]:
# compare Lasso and full model performance
comparison = pd.DataFrame({
    "Model": [
        "Full Features - Linear Regression",
        "Lasso Selected"
    ],
    "MAE": [
        full_mae,
        lasso_mae
    ],
    "RMSE": [
        full_rmse,
        lasso_rmse
    ],
    "R2": [
        full_r2,
        lasso_r2
    ]
})

display(comparison)

,Model,MAE,RMSE,R2
0,Full Features - Linear Regression,3.833560,5.584395,0.188962
1,Lasso Selected,3.776838,5.538090,0.202356


## 4. Extract features after lasso

In [28]:
# Map encoded features back to original features

feature_map = {}

# Numeric features
for col in numeric_features:
    feature_map[f"num__{col}"] = col


# Categorical features
ohe = (
    lasso_pipeline
    .named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["onehot"]
)

encoded_cat_names = ohe.get_feature_names_out(
    categorical_features
)

idx = 0

for original_col, categories in zip(
    categorical_features,
    ohe.categories_
):
    for _ in categories:
        encoded_name = f"cat__{encoded_cat_names[idx]}"
        feature_map[encoded_name] = original_col
        idx += 1


selected_encoded_features["original_feature"] = (
    selected_encoded_features["feature"]
    .map(feature_map)
)

In [29]:
final_features = (
    selected_encoded_features[
        "original_feature"
    ]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(
    "Number of final original features:",
    len(final_features)
)

print("\nFinal Features:")

for feature in final_features:
    print(feature)

Number of final original features: 57

Final Features:
seller_past_deliveries
customer_state
seller_past_orders
product_category
primary_seller_state
customer_region
region_pair
estimated_min_distance_km
seller_past_delivery_days
purchase_month_cos
primary_seller_region
payment_type
weekend_days_to_estimate
same_state
platform_orders_7d
purchase_month_sin
seller_past_late_rate
any_seller_same_state
remote_flag
peak_season_flag
mean_seller_zip_lng
customer_is_capital
max_item_weight_g
mean_seller_zip_lat
multi_seller_flag
days_to_holiday
item_count
payment_missing
longest_side_cm
purchase_weekday
platform_load_ratio
customer_zip_density
seller_count
geolocation_missing
payment_installments
promised_days_per_km
total_weight_g
mean_item_price
seller_backlog_7d
density_g_cm3
same_city
billable_weight_kg
product_category_count
customer_zip_lng
purchase_hour
used_voucher
unique_product_count
customer_zip_lat
volumetric_weight_kg
payment_per_installment
seller_state_count
total_volume_cm3
pro

In [30]:
X_train_final = train_df[
    final_features
].copy()

y_train_final = train_df[
    "delivery_days"
].copy()


X_val_final = val_df[
    final_features
].copy()

y_val_final = val_df[
    "delivery_days"
].copy()


X_test_final = test_df[
    final_features
].copy()

y_test_final = test_df[
    "delivery_days"
].copy()

print("Train X:", X_train_final.shape)
print("Validation X:", X_val_final.shape)
print("Test X:", X_test_final.shape)

Train X: (83947, 57)
Validation X: (6156, 57)
Test X: (6351, 57)


In [31]:
output_path = r"D:\NUS Note\IT5006\it5006_project\data\fe_data\final_selected_features.csv"

final_feature_df = pd.DataFrame({
    "feature": final_features
})

final_feature_df.to_csv(
    output_path,
    index=False
)

print("Saved to:", output_path)

Saved to: D:\NUS Note\IT5006\it5006_project\data\fe_data\final_selected_features.csv
